# Lab 09 solution, notebook 1: a chain that returns calendar entries, and a third tool for the agent

The worked answers to the two challenge tasks of `09a_meet_langchain.ipynb`. The setup repeats the notebook's
model, class and tools, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL, reasoning_effort=EFFORT, timeout=60)

## Task 1: A chain that returns calendar entries

The class is the notebook's `Event`.

In [3]:
from pydantic import BaseModel, Field


class Event(BaseModel):
    """One entry for a calendar."""
    title: str = Field(description="What the event is, in a few words")
    day: str
    time: str = Field(description="Start time on the 24-hour clock, like 19:00")
    bring: str = Field(description="What to bring, or an empty string")
    tell: list[str] = Field(description="People to pass the invitation on to")

In the notebook we called `llm.with_structured_output(Event)` on its own. It has `invoke`, so it can be a step of
a chain like any other. The template in front of it has one gap, `{message}`.

In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("Turn this text message into a calendar entry:\n\n{message}")

to_event = prompt | llm.with_structured_output(Event)

texts = [
    "Football Saturday 6pm at the school pitch, bring water, tell Omar and Lina",
    "Lina's graduation Sunday 11am, bring flowers",
]

for text in texts:
    print(repr(to_event.invoke({"message": text})))

Event(title='Football', day='Saturday', time='18:00', bring='Water', tell=['Omar', 'Lina'])
Event(title="Lina's graduation", day='Sunday', time='11:00', bring='Flowers', tell=[])


Two `Event` objects. The first has the time `18:00` and `tell=['Omar', 'Lina']`: two names in the message, two
items in the list. The second has `11:00` and an empty `tell`, because that message asks us to tell nobody. An
empty list is a valid `list[str]`.

The chain's input is a dictionary, as in Part A, and its output is an `Event`, as in Part B: the last step decides
what a chain returns.

## Task 2: Give the agent a third tool

The two tools of the notebook, and a new one, `get_hobby`, that reads a second small dictionary.

In [5]:
from datetime import date

BIRTHDAYS = {"Reem": "14 March", "Sami": "2 July", "Lina": "30 November"}
HOBBIES = {"Reem": "hiking", "Sami": "chess", "Lina": "baking"}


def get_birthday(name: str) -> str:
    """Look up a friend's birthday in my contacts. Returns the day and the month."""
    print(f"TOOL CALLED: get_birthday({name})")
    return BIRTHDAYS.get(name, f"No birthday saved for {name}")


def days_until(day: int, month: int) -> int:
    """Count the days from today until the next time this day and month come around."""
    print(f"TOOL CALLED: days_until({day}, {month})")
    today = date.today()
    target = date(today.year, month, day)
    if target < today:
        target = date(today.year + 1, month, day)
    return (target - today).days


def get_hobby(name: str) -> str:
    """Look up what a friend likes to do in their free time."""
    print(f"TOOL CALLED: get_hobby({name})")
    return HOBBIES.get(name, f"No hobby saved for {name}")

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    [get_birthday, days_until, get_hobby],
    system_prompt="You help me keep track of my friends' birthdays. Answer in one or two short sentences.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Reem's birthday is coming up. How many days do I have, "
                                              "and what could I get her?"}]}
)

print()
requests = [message for message in result["messages"] if message.type == "ai" and message.tool_calls]
for number, message in enumerate(requests, start=1):
    print(f"reply {number} asks for:", [(call["name"], call["args"]) for call in message.tool_calls])
print()
print(result["messages"][-1].text)

TOOL CALLED: get_birthday(Reem)
TOOL CALLED: days_until(14, 3)
TOOL CALLED: get_hobby(Reem)

reply 1 asks for: [('get_birthday', {'name': 'Reem'})]
reply 2 asks for: [('days_until', {'day': 14, 'month': 3}), ('get_hobby', {'name': 'Reem'})]

Reem’s birthday is in 123 days. Since she enjoys hiking, you could get her a useful hiking accessory, like a lightweight water bottle or a cozy pair of hiking socks.


Three `TOOL CALLED` lines. In our run the model asked for `get_birthday` in its first reply, and for `days_until`
and `get_hobby` together in its second. The answer uses all three results: 123 days, and a gift for someone who
likes hiking.

`days_until` has to come after `get_birthday`, because its two arguments are in the birthday. `get_hobby` depends
on nothing, so the model may ask for it in any reply.

**Key lesson:** adding a tool was one function and one more item in the list. We changed nothing in the loop, and
we never said when to use `get_hobby`: the model read its description and decided.